# 🧠 Advanced Fintech Engineering & Quantitative Pandas Techniques

Welcome to the **Staff-Level Advanced Pandas Techniques Reference Notebook**.

This notebook provides production-grade code patterns, mathematical explanations, and implementation variations for the most advanced data engineering and quantitative analytics techniques demanded by tier-1 fintech institutions.

---

### 📚 Core Advanced Concept Modules:
1. [Multi-Format Enterprise File Ingestion Engine](#1-multi-format-enterprise-file-ingestion-engine) (`.tsv`, `.psv`, `.dat`, `.jsonl`, `.xml`, `.yaml`, `.json`)
2. [Advanced Window & Temporal Momentum Analytics](#2-advanced-window--temporal-momentum-analytics) (`.rolling()`, `.ewm()`, `.shift()`, `.diff()`, `.pct_change()`)
3. [Island-and-Gap Consecutive Streak Tracking](#3-island-and-gap-consecutive-streak-tracking) (`(cond != cond.shift()).cumsum()`)
4. [Markov Transition & Cross-Tabulation Matrices](#4-markov-transition--cross-tabulation-matrices) (`pd.cut()`, `pd.crosstab(normalize='index')`)
5. [Vectorized Policy Engines & Multi-Branch Logic](#5-vectorized-policy-engines--multi-branch-logic) (`np.select()`, `np.where()`, `.rank(method='dense')`)

In [ ]:
# Setup environment and load libraries
import pandas as pd
import numpy as np
import json
import yaml
import xml.etree.ElementTree as ET
import os

DATA_DIR = 'data' if os.path.exists('data') else '../data'
print("✅ Environment Active. Datasets available in:", DATA_DIR)

---
# 1. Multi-Format Enterprise File Ingestion Engine

Real-world fintech systems rarely use clean single-table CSVs. Here are the 7 core production parsing recipes:

In [ ]:
# 🔹 1.1 Ingesting Tab-Separated Values (.tsv)
fx_df = pd.read_csv(os.path.join(DATA_DIR, 'fx_rates_daily.tsv'), sep='\t')
print("TSV FX Rates Shape:", fx_df.shape)
fx_df.head(2)

In [ ]:
# 🔹 1.2 Ingesting Pipe-Separated Values with Comment Headers (.psv)
kyc_df = pd.read_csv(os.path.join(DATA_DIR, 'kyc_audit_records.psv'), sep='|', comment='#')
print("PSV KYC Audit Shape:", kyc_df.shape)
kyc_df.head(2)

In [ ]:
# 🔹 1.3 Ingesting NACHA Fixed-Width Files (.dat) via pd.read_fwf()
col_specs = [(0, 19), (19, 30), (30, 42), (42, 57), (57, 69), (69, 100)]
col_names = ['batch_id', 'customer_id', 'trans_type', 'settlement_usd', 'routing_num', 'account_status']

ach_df = pd.read_fwf(
    os.path.join(DATA_DIR, 'ach_clearing_settlement.dat'),
    colspecs=col_specs,
    names=col_names,
    skiprows=1
)
print("Fixed-Width ACH Clearing Shape:", ach_df.shape)
ach_df.head(2)

In [ ]:
# 🔹 1.4 Ingesting Streaming Newline-Delimited JSON (.jsonl)
telemetry_df = pd.read_json(os.path.join(DATA_DIR, 'device_telemetry.jsonl'), lines=True)
print("Streaming JSONL Telemetry Shape:", telemetry_df.shape)
telemetry_df.head(2)

In [ ]:
# 🔹 1.5 Ingesting Nested REST API Webhook Logs (.json) via pd.json_normalize()
with open(os.path.join(DATA_DIR, 'api_event_logs.json'), 'r') as f:
    raw_api_payload = json.load(f)

api_events_df = pd.json_normalize(raw_api_payload['data'])
print("Flattened Nested JSON Shape:", api_events_df.shape)
api_events_df[['event_id', 'customer_id', 'client_info.ip_address', 'security_flags.risk_score']].head(2)

In [ ]:
# 🔹 1.6 Ingesting Credit Bureau XML Trees (.xml) via xml.etree.ElementTree
tree = ET.parse(os.path.join(DATA_DIR, 'credit_bureau_scores.xml'))
root = tree.getroot()

bureau_records = []
for report in root.findall('report'):
    bureau_records.append({
        'pull_id': report.findtext('pull_id'),
        'customer_id': report.findtext('customer_id'),
        'bureau_name': report.findtext('bureau_name'),
        'fico_score': int(report.findtext('fico_score_8') or 0),
        'delinquency_24m': int(report.findtext('delinquency_count_24m') or 0),
        'revolving_util_pct': float(report.findtext('revolving_utilization_pct') or 0.0),
        'hard_inquiries_12m': int(report.findtext('hard_inquiries_12m') or 0)
    })

bureau_df = pd.DataFrame(bureau_records)
print("Parsed XML Credit Bureau Shape:", bureau_df.shape)
bureau_df.head(2)

In [ ]:
# 🔹 1.7 Ingesting Hierarchical YAML Watchlists (.yaml) via yaml.safe_load()
with open(os.path.join(DATA_DIR, 'aml_sanctions_watchlist.yaml'), 'r') as f:
    sanctions_cfg = yaml.safe_load(f)['sanctioned_entities']

sanctioned_countries = {c for e in sanctions_cfg for c in e.get('high_risk_countries', [])}
sanctioned_ips = tuple(ip for e in sanctions_cfg for ip in e.get('linked_ip_prefixes', []))

print("Extracted Sanctioned Countries:", sanctioned_countries)
print("Extracted Sanctioned IP Subnets:", sanctioned_ips)

---
# 2. Advanced Window & Temporal Momentum Analytics

Temporal analysis requires rolling windows, moving averages, and momentum rates-of-change.

In [ ]:
# 🔹 2.1 Multi-Window Time Series: Rolling Sum, EWMA, and Differencing
tx = pd.read_csv(os.path.join(DATA_DIR, 'raw_transactions.csv'))
tx['tx_date'] = pd.to_datetime(tx['transaction_date'], errors='coerce').dt.normalize()
tx['amount'] = pd.to_numeric(tx['transaction_amount'], errors='coerce').fillna(0.0)

# Aggregate to Daily Region Grain
daily_ts = (
    tx[tx['transaction_status'] == 'Completed']
    .groupby(['region', 'tx_date'], as_index=False)['amount']
    .sum()
    .sort_values(by=['region', 'tx_date'])
    .reset_index(drop=True)
)

# Compute Grouped Window Functions
daily_ts['rolling_7d_vol'] = daily_ts.groupby('region')['amount'].transform(lambda s: s.rolling(7, min_periods=1).sum())
daily_ts['ewma_14d_vol'] = daily_ts.groupby('region')['amount'].transform(lambda s: s.ewm(span=14).mean())
daily_ts['day_over_day_change_pct'] = daily_ts.groupby('region')['amount'].transform(lambda s: s.pct_change() * 100)
daily_ts['acceleration_delta'] = daily_ts.groupby('region')['amount'].transform(lambda s: s.diff())

daily_ts.dropna().head(5)

---
# 3. Island-and-Gap Consecutive Streak Tracking

> **The Island-and-Gap Formula:** `streak_block = (condition != condition.shift()).cumsum()`

This algorithm identifies consecutive sequences of events (e.g. 3 consecutive failed logins or consecutive card declines) without using slow Python `for` loops.

In [ ]:
# 🔹 3.1 Tracking Consecutive Failed MFA Streaks per Customer
events = api_events_df.sort_values(by=['customer_id', 'timestamp']).copy()
events['is_mfa_failed'] = (events['security_flags.mfa_prompted'] == True) & (events['security_flags.mfa_passed'] == False)

# Step 1: Create unique block ID whenever state flips
events['streak_block'] = (events['is_mfa_failed'] != events.groupby('customer_id')['is_mfa_failed'].shift()).cumsum()

# Step 2: Compute consecutive streak length within each failed block
events['consecutive_failures'] = (
    events[events['is_mfa_failed']]
    .groupby(['customer_id', 'streak_block'])
    .cumcount() + 1
)
events['consecutive_failures'] = events['consecutive_failures'].fillna(0).astype(int)

# Isolate active brute-force sessions (>= 2 consecutive failures)
brute_force_attacks = events[events['consecutive_failures'] >= 2]
print(f"Flagged {len(brute_force_attacks)} brute force streak events across {brute_force_attacks['customer_id'].nunique()} accounts.")
brute_force_attacks[['customer_id', 'event_id', 'timestamp', 'consecutive_failures']].head()

---
# 4. Markov Transition & Cross-Tabulation Matrices

Cross-tabulation matrices measure portfolio migration across discrete rating bands.

In [ ]:
# 🔹 4.1 Continuous Binning (pd.cut) & Row-Normalized Transition Matrix (pd.crosstab)
cust = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
cust_bureau = cust.merge(bureau_df, on='customer_id', how='inner')

# Discretize FICO Score into Underwriting Bands
bins = [0, 580, 670, 740, 850]
labels = ['Subprime (<580)', 'Near-Prime [580-669]', 'Prime [670-739]', 'Super-Prime [740-850]']
cust_bureau['fico_band'] = pd.cut(cust_bureau['fico_score'], bins=bins, labels=labels, right=False)

# Normalized Transition Matrix: Internal Risk Tier vs Bureau Score Band
transition_matrix = pd.crosstab(
    cust_bureau['risk_tier'],
    cust_bureau['fico_band'],
    normalize='index'
).round(4) * 100

print("Underwriting Portfolio Risk Migration Matrix (% Distribution per Risk Tier):")
display(transition_matrix)

---
# 5. Vectorized Policy Engines & Multi-Branch Logic

Vectorized engines evaluate complex multi-tier business rules in a single high-speed pass using `np.select` and dense group ranking.

In [ ]:
# 🔹 5.1 Vectorized Multi-Branch Policy Action Engine (np.select)
merchants = pd.read_csv(os.path.join(DATA_DIR, 'merchants.csv'))

# Vectorized Underwriting Action Rules
conditions = [
    (merchants['monthly_volume_est'] > 1000000) & (merchants['risk_rating'].isin(['High', 'Extreme'])),
    (merchants['is_chargeback_monitored'] == 1) | (merchants['risk_rating'] == 'Extreme'),
    (merchants['risk_rating'] == 'Moderate')
]

actions = [
    'IMMEDIATE_SETTLEMENT_HOLD',
    '7_DAY_ROLLING_RESERVE',
    'STANDARD_MONITORING'
]

merchants['underwriting_policy'] = np.select(conditions, actions, default='AUTO_PAYOUT_CLEARED')

# Grouped Dense Ranking: Rank top volume merchants within each category
merchants['category_vol_rank'] = merchants.groupby('category')['monthly_volume_est'].rank(method='dense', ascending=False).astype(int)

display(merchants[['merchant_id', 'category', 'risk_rating', 'monthly_volume_est', 'underwriting_policy', 'category_vol_rank']].head(10))